# Data Cleaning Pipeline
Optimized preprocessing pipeline for Bayut transactions dataset.

In [67]:
#importing libraries
import pandas as pd
import numpy as np

In [68]:
# Load Dataset
df = pd.read_csv('bayut_all_locations_transactions.csv')

In [69]:
df.columns

Index(['main_location', 'sub_location', 'building_project',
       'location_selected', 'date', 'category', 'sub-category',
       'location_image', 'is_off_plan?', 'price(EAD)',
       'is_Vacant_at_time_of_sale?', 'type', 'beds', 'built_up(sqft)',
       'plot(sqft)'],
      dtype='str')

In [70]:
df.shape

(162473, 15)

In [71]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 162473 entries, 0 to 162472
Data columns (total 15 columns):
 #   Column                      Non-Null Count   Dtype
---  ------                      --------------   -----
 0   main_location               162473 non-null  str  
 1   sub_location                161525 non-null  str  
 2   building_project            55605 non-null   str  
 3   location_selected           162473 non-null  str  
 4   date                        162473 non-null  str  
 5   category                    162473 non-null  str  
 6   sub-category                162473 non-null  str  
 7   location_image              162473 non-null  str  
 8   is_off_plan?                162473 non-null  bool 
 9   price(EAD)                  162473 non-null  str  
 10  is_Vacant_at_time_of_sale?  162473 non-null  bool 
 11  type                        162473 non-null  str  
 12  beds                        161955 non-null  str  
 13  built_up(sqft)              162473 non-null  str  
 14 

In [72]:
# Drop Unnecessary Columns
df.drop(columns=['location_selected', 'category', 'sub-category', 'location_image'], inplace=True)

In [73]:
# Date Features
df['date'] = pd.to_datetime(df['date'])

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter

/tmp/ipykernel_38815/1525606847.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])


In [74]:
# Numeric Cleaning
num_cols = ['price(EAD)', 'built_up(sqft)', 'plot(sqft)']

for col in num_cols:
    df[col] = df[col].astype(str).str.replace(',', '', regex=False)

df['price(EAD)'] = pd.to_numeric(df['price(EAD)'], errors='coerce')
df['built_up(sqft)'] = pd.to_numeric(df['built_up(sqft)'], errors='coerce')
df['plot(sqft)'] = pd.to_numeric(df['plot(sqft)'], errors='coerce')

In [75]:
df.isna().sum()

main_location                      0
sub_location                     948
building_project              106868
date                               0
is_off_plan?                       0
price(EAD)                         0
is_Vacant_at_time_of_sale?         0
type                               0
beds                             518
built_up(sqft)                   564
plot(sqft)                    140934
year                               0
month                              0
quarter                            0
dtype: int64

In [76]:
# Remove Rows Without Area Info
df = df[~(df['built_up(sqft)'].isna() & df['plot(sqft)'].isna())]

In [77]:
# Fix Missing Values Based on Property Type

In [78]:
# Residential plots have no built_up area
df.loc[(df['type'] == 'Residential Plot') & (df['built_up(sqft)'].isna()), 'built_up(sqft)'] = 0

In [79]:
# Apartments / similar properties have no plot area
plot_zero_types = ['Apartment', 'Hotel Apartment', 'Penthouse']

df.loc[(df['type'].isin(plot_zero_types)) & (df['plot(sqft)'].isna()), 'plot(sqft)'] = 0

In [80]:

# Fill Building Project
df['building_project'] = df['building_project'].fillna(df['sub_location'])

In [81]:
# Fix Location Missing Values
mask = df['sub_location'].isna() & df['building_project'].isna()

df.loc[mask, 'sub_location'] = df.loc[mask, 'main_location']
df.loc[mask, 'building_project'] = df.loc[mask, 'main_location']


In [82]:
# Beds Cleaning
df.loc[(df['type'] == 'Residential Plot') & (df['beds'].isna()), 'beds'] = '0'

In [83]:
#removed rows which are having having na values in columns built_up or beds
removed_rows = pd.DataFrame()

rows_with_missing_area_or_beds = df[df['built_up(sqft)'].isna() | df['beds'].isna()]

removed_rows = pd.concat([removed_rows, rows_with_missing_area_or_beds], ignore_index=True)

df.drop(rows_with_missing_area_or_beds.index, inplace=True)

In [84]:
df['price_per_sqft'] = np.where(
    df['type'] == 'Residential Plot',
    df['price(EAD)'] / df['plot(sqft)'],
    df['price(EAD)'] / df['built_up(sqft)']
)

In [85]:
price_per_sqft_by_location = (
    df.groupby(['year', 'quarter','sub_location'])['price_per_sqft']
    .median()
    .reset_index()
)

In [86]:
price_per_sqft_by_location_type = (
    df.groupby(['year', 'quarter','sub_location', 'type'])['price_per_sqft']
    .median()
    .reset_index()
)

In [87]:
price_per_sqft_by_location_type_beds = (
    df.groupby(['year', 'quarter','sub_location', 'type', 'beds'])['price_per_sqft']
    .median()
    .reset_index()
)

In [88]:
price_per_sqft_by_location_type_beds_time = (
    df.groupby(['year', 'quarter', 'sub_location', 'type', 'beds'])['price_per_sqft']
    .median()
    .reset_index()
)

In [89]:
df[df['sub_location']=='JVC District 10']

,main_location,sub_location,building_project,date,is_off_plan?,price(EAD),is_Vacant_at_time_of_sale?,type,beds,built_up(sqft),plot(sqft),year,month,quarter,price_per_sqft
0,Jumeirah Village Circle (JVC),JVC District 10,10 Oxford,2025-09-10,True,1220000,False,Apartment,1,814.0,0.0,2025,9,3,1498.771499
1,Jumeirah Village Circle (JVC),JVC District 10,10 Oxford,2026-02-11,True,740000,False,Apartment,Studio,409.0,0.0,2026,2,1,1809.290954
2,Jumeirah Village Circle (JVC),JVC District 10,10 Oxford,2025-08-12,True,2165400,False,Apartment,2,1608.0,0.0,2025,8,3,1346.641791
3,Jumeirah Village Circle (JVC),JVC District 10,10 Oxford,2025-12-12,True,775000,False,Apartment,Studio,423.0,0.0,2025,12,4,1832.151300
4,Jumeirah Village Circle (JVC),JVC District 10,10 Oxford,2026-02-12,True,735000,False,Apartment,Studio,417.0,0.0,2026,2,1,1762.589928
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2699,Jumeirah Village Circle (JVC),JVC District 10,Vitality Residence,2025-09-09,True,1299893,False,Apartment,1,1022.0,0.0,2025,9,3,1271.910959
2700,Jumeirah Village Circle (JVC),JVC District 10,Westar La Residencia del Sol,2025-07-21,False,2500000,True,Villa,3,3812.0,3812.0,2025,7,3,655.823715
2701,Jumeirah Village Circle (JVC),JVC District 10,Westar La Residencia del Sol,2025-09-22,False,2650000,True,Villa,3,3515.0,3515.0,2025,9,3,753.911807
2702,Jumeirah Village Circle (JVC),JVC District 10,Westar La Residencia del Sol,2025-12-08,False,2500000,True,Villa,3,3619.0,3619.0,2025,12,4,690.798563


In [90]:
price_per_sqft_by_location[price_per_sqft_by_location['sub_location']=='JVC District 10']

,year,quarter,sub_location,price_per_sqft
394,2025,1,JVC District 10,1399.742931
1423,2025,2,JVC District 10,1384.614835
2614,2025,3,JVC District 10,1409.114582
3862,2025,4,JVC District 10,1521.948179
5151,2026,1,JVC District 10,1500.916555


In [91]:
price_per_sqft_by_location_type[price_per_sqft_by_location_type['sub_location']=='JVC District 10']

,year,quarter,sub_location,type,price_per_sqft
411,2025,1,JVC District 10,Apartment,1388.644928
412,2025,1,JVC District 10,Hotel Apartment,1564.942169
413,2025,1,JVC District 10,Villa,660.241013
1499,2025,2,JVC District 10,Apartment,1378.576105
1500,2025,2,JVC District 10,Hotel Apartment,1673.732962
1501,2025,2,JVC District 10,Villa,963.241213
2756,2025,3,JVC District 10,Apartment,1407.200000
2757,2025,3,JVC District 10,Hotel Apartment,1812.366738
2758,2025,3,JVC District 10,Villa,788.000051
4075,2025,4,JVC District 10,Apartment,1516.667964


In [92]:
price_per_sqft_by_location_type_beds[price_per_sqft_by_location_type_beds['sub_location']=='JVC District 10']

,year,quarter,sub_location,type,beds,price_per_sqft
806,2025,1,JVC District 10,Apartment,1,1336.418991
807,2025,1,JVC District 10,Apartment,2,1146.912224
808,2025,1,JVC District 10,Apartment,3,894.503985
809,2025,1,JVC District 10,Apartment,Studio,1648.402508
810,2025,1,JVC District 10,Hotel Apartment,1,1557.285624
811,2025,1,JVC District 10,Hotel Apartment,Studio,1929.460581
812,2025,1,JVC District 10,Villa,3,655.457395
813,2025,1,JVC District 10,Villa,4,665.024631
3170,2025,2,JVC District 10,Apartment,1,1352.304611
3171,2025,2,JVC District 10,Apartment,2,1294.517920


In [98]:
price_per_sqft_by_location = price_per_sqft_by_location.rename(
    columns={'price_per_sqft': 'median_price_sqft_location'}
)

price_per_sqft_by_location_type = price_per_sqft_by_location_type.rename(
    columns={'price_per_sqft': 'median_price_sqft_location_type'}
)

price_per_sqft_by_location_type_beds = price_per_sqft_by_location_type_beds.rename(
    columns={'price_per_sqft': 'median_price_sqft_location_type_beds'}
)

price_per_sqft_by_location_type_beds_time = price_per_sqft_by_location_type_beds_time.rename(
    columns={'price_per_sqft': 'median_price_sqft_location_type_beds_time'}
)

In [99]:
merged_price_data = price_per_sqft_by_location_type_beds_time.merge(
    price_per_sqft_by_location_type_beds,
    on=['year','quarter','sub_location','type','beds'],
    how='left'
)

merged_price_data = merged_price_data.merge(
    price_per_sqft_by_location_type,
    on=['year','quarter','sub_location','type'],
    how='left'
)

merged_price_data = merged_price_data.merge(
    price_per_sqft_by_location,
    on=['year','quarter','sub_location'],
    how='left'
)

In [100]:
merged_price_data

,year,quarter,sub_location,type,beds,median_price_sqft_location_type_beds_time,median_price_sqft_location_type_beds,median_price_sqft_location_type,median_price_sqft_location
0,2025,1,11 Hills Park,Apartment,1,1526.739078,1526.739078,1504.108303,1504.108303
1,2025,1,11 Hills Park,Apartment,Studio,1485.556468,1485.556468,1504.108303,1504.108303
2,2025,1,15 Northside,Apartment,1,2512.998267,2512.998267,2512.998267,2512.998267
3,2025,1,15 Northside,Apartment,Studio,2359.070655,2359.070655,2512.998267,2512.998267
4,2025,1,17 Icon Bay,Apartment,2,1949.860724,1949.860724,1949.860724,1949.860724
...,...,...,...,...,...,...,...,...,...
14200,2026,1,Zenith Towers,Apartment,2,805.025162,805.025162,805.025162,805.025162
14201,2026,1,Zephyra Residences,Apartment,1,2152.614204,2152.614204,2152.614204,2152.614204
14202,2026,1,Zephyra Residences,Apartment,2,2061.077148,2061.077148,2152.614204,2152.614204
14203,2026,1,Zinnia,Villa,3,964.765101,964.765101,964.765101,964.765101
